# SOCCAT — Annotation Conversion

Converts `data/manual_annotations/annotations_09_10_2025.csv` into one NLI pair CSV per category, ready for `cv_pipeline.py` and `replicate_from_hub.py`.

**Output format per row:**

| Column | Description |
|---|---|
| `sentence_id` | Unique sentence identifier |
| `premise` | The original sentence |
| `hypothesis` | `This sentence refers to [category] as a social group, specifically "[label]".` |
| `nli_label` | `0` = entailment (sentence mentions this group), `1` = not_entailment |
| `hypothesis_label` | The specific label (e.g. `tenants`) |
| `outlet` | News outlet |
| `country` | Germany / France |
| `date` | ISO date string |
| `year` | 4-digit year |

Sentences with multiple categories appear in every relevant category CSV. Unmapped labels (`others`, `enterprises`, `volunteers`, etc.) are treated as negatives across all categories.

## 1. Install & Import

In [ ]:
%%capture
!pip install pandas -U

In [ ]:
import os
import pandas as pd

## 2. Configuration

Set `INPUT_FILE` to your annotations CSV and `OUTPUT_DIR` to where the NLI pair CSVs should be written.

In [ ]:
INPUT_FILE = "../../data/manual_annotations/annotations_ground_truth.csv"
OUTPUT_DIR = "nli_pairs_by_category"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Taxonomy

All 8 categories and their labels. Edit here if the taxonomy changes.

In [ ]:
TAXONOMY = {
    "socio_economic_position": [
        "lower class", "middle class", "upper class",
        "capital owners, investors and shareholders",
        "unskilled or unqualified", "skilled or qualified",
    ],
    "labor_market_position": [
        "wage and salary earners", "civil servants", "CEOs and corporate leaders",
        "employers", "entrepreneurs", "self-employed and freelancers",
        "unemployed", "retirees", "housewives and househusbands",
    ],
    "age_and_family_status": [
        "parents and families",
        "minors, including children and pupils",
        "youth, including students and apprentices",
        "middle-aged and pre-retirement age groups",
        "elderly", "couples", "singles",
    ],
    "identities": [
        "men", "women", "cisgender and heterosexuals", "LGBTQIA+",
        "disabled people",
        "people with an immigration background, including immigrants",
        "Ethnic and racial minorities", "Christians", "Jews", "Muslims",
        "multiple (or other) religious or minority groups",
    ],
    "profession": [
        "athletes", "authors and artists", "doctors", "farmers and fishermen",
        "health and care professionals", "journalists", "legal professionals",
        "politicians and high-ranking officials", "sex workers",
        "scientists and professors", "security forces", "soldiers",
        "teachers and educators", "other professions",
    ],
    "social_roles_and_behavior": [
        "consumers and clients", "car drivers", "patients",
    ],
    "social_deviance": [
        "extremists",
        "terrorists, rebels, revolutionaries and/or movements of armed resistance",
        "offenders, criminals, prisoners and/or accused people",
        "drug addicts",
    ],
    "real_estate_ownership": [
        "real-estate owners", "tenants", "homeless",
    ],
}

## 4. Label Mapping

Maps raw values from `specific_group_new` to exact taxonomy labels. Anything not listed here is treated as negative for all categories.

In [ ]:
LABEL_MAP = {
    # socio_economic_position
    "lower class":                          "lower class",
    "middle class":                         "middle class",
    "upper class":                          "upper class",
    "capital owners, investors and shareholders": "capital owners, investors and shareholders",
    "unskilled or unqualified":             "unskilled or unqualified",
    "skilled or qualified":                 "skilled or qualified",
    # labor_market_position
    "wage and salary earners":              "wage and salary earners",
    "civil servants":                       "civil servants",
    "ceos and corporate leaders":           "CEOs and corporate leaders",
    "employers":                            "employers",
    "entrepreneurs":                        "entrepreneurs",
    "entrepreneurs (smes)":                 "entrepreneurs",
    "entrepreneurs in [specific] sector":   "entrepreneurs",
    "entrepreneurs (large enterprises)":    "CEOs and corporate leaders",
    "self-employed and freelancers":        "self-employed and freelancers",
    "unemployed":                           "unemployed",
    "retirees":                             "retirees",
    "housewife and househusband":           "housewives and househusbands",
    "housewives and househusbands":         "housewives and househusbands",
    # age_and_family_status
    "parents and families":                 "parents and families",
    "minors":                               "minors, including children and pupils",
    "minors, including children and pupils": "minors, including children and pupils",
    "youth":                                "youth, including students and apprentices",
    "youth, including students and apprentices": "youth, including students and apprentices",
    "middle-aged and pre-retirement age groups": "middle-aged and pre-retirement age groups",
    "elderly":                              "elderly",
    "couples":                              "couples",
    "singles":                              "singles",
    # identities
    "men":                                  "men",
    "women":                                "women",
    "cisgender and heterosexuals":          "cisgender and heterosexuals",
    "lgbtqia+":                             "LGBTQIA+",
    "lgbtqqia+":                            "LGBTQIA+",
    "disabled people":                      "disabled people",
    "people with an immigration background, including immigrants":
        "people with an immigration background, including immigrants",
    "ethnic and racial minorities":         "Ethnic and racial minorities",
    "christians":                           "Christians",
    "jews":                                 "Jews",
    "muslims":                              "Muslims",
    "multiple (or other specific) religious or minority groups":
        "multiple (or other) religious or minority groups",
    "multiple (or other) religious or minority groups":
        "multiple (or other) religious or minority groups",
    # profession
    "athletes":                             "athletes",
    "authors and artists":                  "authors and artists",
    "doctors":                              "doctors",
    "farmers and fishermen":                "farmers and fishermen",
    "health and care professionals":        "health and care professionals",
    "journalists":                          "journalists",
    "legal professionals":                  "legal professionals",
    "politicians and high-ranking officials": "politicians and high-ranking officials",
    "sex workers":                          "sex workers",
    "prostitutes":                          "sex workers",
    "scientists and professors":            "scientists and professors",
    "security forces":                      "security forces",
    "soldiers":                             "soldiers",
    "teachers and educators":               "teachers and educators",
    "other profession":                     "other professions",
    "other professions":                    "other professions",
    # social_roles_and_behavior
    "consumers and clients":                "consumers and clients",
    "car drivers":                          "car drivers",
    "patients":                             "patients",
    # social_deviance
    "extremists":                           "extremists",
    "terrorists":                           "terrorists, rebels, revolutionaries and/or movements of armed resistance",
    "terrorists, rebels, revolutionaries and/or movements of armed resistance":
        "terrorists, rebels, revolutionaries and/or movements of armed resistance",
    "offenders, criminals, prisoners and/or accused people":
        "offenders, criminals, prisoners and/or accused people",
    "drug addicts":                         "drug addicts",
    # real_estate_ownership
    "real-estate owner":                    "real-estate owners",
    "real-estate owners":                   "real-estate owners",
    "real estate owners":                   "real-estate owners",
    "tenants":                              "tenants",
    "homeless":                             "homeless",
}

## 5. Helper Functions

In [ ]:
def make_hypothesis(category, label):
    cat_display = {
        "socio_economic_position":   "socio-economic position",
        "labor_market_position":     "labor market position",
        "age_and_family_status":     "age and family status",
        "identities":                "identities and minority/majority status",
        "profession":                "profession",
        "social_roles_and_behavior": "social roles and behavior",
        "social_deviance":           "social deviance",
        "real_estate_ownership":     "real estate ownership",
    }[category]
    return (
        f'This sentence refers to {cat_display} as a social group, '
        f'specifically "{label}".'
    )


def parse_labels(raw):
    """Semicolon-separated specific_group_new → set of taxonomy labels."""
    if pd.isna(raw):
        return set()
    mapped = set()
    for part in str(raw).split(";"):
        new_lbl = LABEL_MAP.get(part.strip().lower())
        if new_lbl:
            mapped.add(new_lbl)
    return mapped

## 6. Load Data

In [ ]:
df = pd.read_csv(INPUT_FILE)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df["_labels"] = df["specific_group_new"].apply(parse_labels)

print(f"Sentences: {len(df):,}")
print(f"Outlets:   {sorted(df['outlet'].dropna().unique())}")
print(f"Years:     {int(df['year'].min())}–{int(df['year'].max())}")
print(f"Countries: {df['country'].value_counts().to_dict()}")

# Show unmapped labels
unmapped = {}
for raw in df["specific_group_new"].dropna():
    for part in str(raw).split(";"):
        key = part.strip().lower()
        if key and not LABEL_MAP.get(key):
            unmapped[key] = unmapped.get(key, 0) + 1

print("\nUnmapped labels (treated as negatives):")
for lbl, cnt in sorted(unmapped.items(), key=lambda x: -x[1]):
    print(f"  {cnt:5d}×  {lbl}")

## 7. Generate & Save NLI Pairs

One CSV per category. Each sentence appears once per label in that category. Sentences with multiple categories appear in every relevant file.

In [ ]:
summary = []

for cat, labels in TAXONOMY.items():
    rows = []
    for _, row in df.iterrows():
        pos_labels = row["_labels"]
        for label in labels:
            rows.append({
                "sentence_id":      int(row["id"]),
                "premise":          str(row["text"]).strip(),
                "hypothesis":       make_hypothesis(cat, label),
                "nli_label":        0 if label in pos_labels else 1,
                "hypothesis_label": label,
                "outlet":           row["outlet"] if pd.notna(row["outlet"]) else "Unknown",
                "country":          row["country"] if pd.notna(row["country"]) else "Unknown",
                "date":             row["date"].date().isoformat() if pd.notna(row["date"]) else None,
                "year":             int(row["year"]) if pd.notna(row["year"]) else None,
            })

    out  = pd.DataFrame(rows)
    path = os.path.join(OUTPUT_DIR, f"nli_dataset_{cat}.csv")
    out.to_csv(path, index=False)

    n_pos = (out["nli_label"] == 0).sum()
    n_neg = (out["nli_label"] == 1).sum()
    pct   = 100 * n_pos / len(out)
    summary.append({"category": cat, "total_pairs": len(out),
                    "positives": n_pos, "negatives": n_neg, "pos_pct": round(pct, 1)})
    print(f"  {cat:35s}  {len(out):7,} pairs  pos={n_pos:5,} ({pct:.1f}%)")

print(f"\nAll files saved to: {OUTPUT_DIR}")

## 8. Summary

In [ ]:
pd.DataFrame(summary).set_index("category")